# 第4周 · 周六：数仓综合练习

> 难度：⭐⭐⭐⭐ | 定位：综合运用本周全部知识，完成一个完整的数仓方案设计和优化
> 前置依赖：周一至周五全部数仓知识 | 后续用途：实际工作中独立设计或评估数仓方案
> 面试地位：常以大作业或案例分析形式出现

---

## 📘 区块1：核心概念与第一性原理

### 1.1 为什么需要综合练习

前五天你学了：数据流转链路 → 数仓分层建模 → SCD与宽表 → Hive大数据处理 → 数据倾斜与优化。

这些知识不是孤立的——它们在真实工作中是协同运作的。今天的综合练习让你在一个完整的案例中看到：分层怎么设计、维度怎么处理、查询怎么优化、倾斜怎么预防——这些决策之间如何互相影响。

### 1.2 案例背景：成长中电商公司的数据困境

你是这家公司第一个专业的数据分析师。入职第一天，CTO告诉你现在的情况：
- 只有一个MySQL数据库支撑全部业务
- 订单表每天新增500万行，已经存了200亿行
- 分析师直接在MySQL上跑SQL——但跑一个"去年同期的品类分析"要20分钟
- 历史数据超过6个月的被手动删除（MySQL空间不够）
- 同一个指标（GMV），运营算的、财务算的、你算的——三个数字都不一样
- 没有数据字典，不知道300多张表哪些能用、哪些是临时表

你的任务：设计数仓方案来系统性解决这些问题。

## 🧩 区块2：知识点详细拆解——完整方案设计

### 2.1 问题诊断

从本周所学知识，我们可以诊断出以下问题：

| 症状 | 根因 | 涉及的知识 |
|------|------|-----------|
| SQL跑20分钟 | 没有分层，每次都要从原始数据算；没有分区 | 周二(分层)、周四(Hive分区) |
| 数据被删 | 没有数仓，OLTP存储有限 | 周一(OLTP vs OLAP) |
| GMV口径不一 | 没有DWD层统一口径 | 周二(分层)、周三(SCD) |
| 找不到能用的表 | 没有数据字典、命名不规范 | 周二(分层命名规范) |
| 分析影响业务 | 分析师在OLTP上跑大查询 | 周一(OLTP vs OLAP分离) |
| 历史分析不准 | 维度被覆盖（如用户城市变更） | 周三(SCD) |

### 2.2 分层设计方案

**ODS层**（操作数据层）——从MySQL同步过来的原始数据：
```
ods_order_log_df        → 原始订单日志（按dt分区，全量快照）
ods_user_info_df        → 原始用户信息
ods_product_info_df     → 原始商品信息
ods_shop_info_df        → 原始店铺信息
ods_payment_log_df      → 原始支付日志
ods_refund_log_df       → 原始退款日志
```
同步策略：Canal监听MySQL binlog → Kafka → Spark写入Hive ODS层
时效性：T+1（每天凌晨3点同步前一天的数据）
存储格式：TEXTFILE（保持原始，不改任何东西）

**DWD层**（明细数据层）——清洗+标准化后的数据：
```
dwd_order_detail_df     → 订单明细（字段标准化、空值处理、口径统一）
dwd_user_register_df    → 用户注册明细
dwd_payment_detail_df   → 支付明细
dwd_refund_detail_df    → 退款明细
dim_user                → 用户维度表（SCD Type 2处理等级和城市）
dim_product             → 商品维度表（SCD Type 2处理品类）
dim_shop                → 店铺维度表
dim_date                → 时间维度表（日/周/月/季/年）
```
清洗内容：
- 字段名统一（uid→user_id, mobile→phone）
- 空值处理（amount的NULL→0, status的NULL→'未知'）
- GMV口径定义：gmv_gross(下单金额)、gmv_net(实付-退款)、gmv_net_30d(扣除30天后退款)
- 手机号格式统一（去+86、去空格、验证11位）
存储格式：ORC（压缩+列式存储，查询快）

**DWS层**（汇总数据层）——轻度聚合：
```
dws_city_category_daily_1d   → 城市×品类日报
dws_user_daily_1d            → 用户日报（当日订单数、金额、品类数）
dws_shop_daily_1d            → 店铺日报
dws_gmv_daily_1d             → GMV日报（总GMV、订单数、客单价）
dws_channel_daily_1d         → 渠道日报
```
汇总原则：只汇总"最常被查询的维度组合"
存储格式：ORC

**ADS层**（应用数据层）——面向具体应用：
```
ads_ceo_dashboard_daily      → CEO看板（核心数字，一行搞定）
ads_operation_report_daily   → 运营日报
ads_finance_monthly          → 财务月报
ads_user_wide                → 用户分析宽表（80列，覆盖80%用户分析）
ads_product_wide             → 商品分析宽表
```
存储格式：ORC

### 2.3 SCD设计决策

| 维度属性 | SCD类型 | 理由 |
|---------|---------|------|
用户会员等级 | Type 2 | 需要看等级变化和升级路径 |
用户所在城市 | Type 2 | 地域分析需要历史归属 |
用户手机号 | Type 1 | 只需要最新手机号做触达 |
商品所属品类 | Type 2 | 品类调整影响历史分析 |
店铺名称 | Type 1 | 偶尔改名，不需要追溯 |
用户头像URL | Type 1 | 分析不关心 |

### 2.4 分区策略

| 表 | 分区字段 | 粒度 | 理由 |
|------|---------|------|------|
ODS/DWD层所有表 | dt（日期） | 天 | 95%查询按天过滤 |
DWS层表 | dt（日期） | 天 | 按天汇总 |
ADS层表 | dt（日期） | 天/月 | 按需 |
dim_维度表 | 不分区（或用ds表示快照日） | - | 维度表数据量小 |

### 2.5 查询性能预估

| 查询场景 | 原MySQL | 数仓+DWD | 数仓+DWS |
|---------|---------|---------|----------|
去年同期的品类分析 | 20分钟 | 1-2分钟 | 0.1秒 |
昨日各城市GMV | 5分钟 | 30秒 | 0.1秒 |
近30天用户分析 | 30分钟 | 3-5分钟 | 0.5秒 |
CEO看板（每日） | 手动跑 | - | 0.1秒（预计算） |

### 2.6 倾斜预防设计

**已知风险点**：
1. 订单表：大促期间大量订单集中在少数热门商品→JOIN倾斜
2. 用户行为表：机器人用户可能占大量行→GROUP BY倾斜
3. 渠道表：NULL channel_id可能很多→GROUP BY倾斜

**预防措施**：
1. DWS层的预聚合本身就是最好的防倾斜（在ETL阶段处理倾斜，分析师直接用汇总结果）
2. DWD层订单表按用户分桶（256桶），减少GROUP BY user_id的倾斜
3. 大促前对核心JOIN做MAPJOIN改造
4. 定期检查所有DWD/DWS表的key分布

## 💻 区块3：动手练习

In [ ]:
# ============================================================
# 综合练习：为电商平台设计完整数仓方案
# ============================================================

print("=" * 60)
print("任务1：分层设计——每层放什么表？")
print("=" * 60)
print()

print("你的答案（参考方案见上面的知识点拆解）：")
print()
print("ODS层需要哪些表？（从MySQL同步过来的原始表）")
print("  1. （填：订单日志？用户信息？商品信息？）")
print("  2. ")
print("  3. ")
print()
print("DWD层需要清洗什么？（至少说3点）")
print("  1. （填：字段统一——比如uid和user_id统一为user_id）")
print("  2. （填：空值处理——比如amount的NULL怎么处理）")
print("  3. （填：口径统一——比如GMV包含什么不包含什么）")
print()
print("DWS层需要哪些汇总表？（至少列出3张）")
print("  1. （填：用户日活表？品类日汇总？渠道日汇总？）")
print("  2. ")
print("  3. ")
print()
print("ADS层需要产出什么？（至少列出2个应用）")
print("  1. （填：CEO看板？运营日报？推荐算法特征表？）")
print("  2. ")
print()

print("=" * 60)
print("任务2：SCD设计")
print("=" * 60)
print()
print("以下属性该用什么SCD类型？为什么？")
print()
scd_design = [
    ("用户会员等级（每月更新）", "需要分析升级路径", "?", "（需要历史→Type 2）"),
    ("商品纠错（名称拼写改正）", "只需要现在的正确名称", "?", "（不需要历史→Type 1）"),
    ("部门改名（偶尔发生）", "只需知道改名前vs改名后", "?", "（偶尔变化+简单对比→Type 3）"),
]
for scenario, req, blank, hint in scd_design:
    print(f"  场景：{scenario}")
    print(f"  需求：{req}")
    print(f"  你的选择：{blank} → 提示：{hint}")
    print()

print("=" * 60)
print("任务3：解决GMV口径不一致问题")
print("=" * 60)
print()
print("问题：")
print("  运营的GMV = 下单金额（含未支付）")
print("  财务的GMV = 实付金额（含退款扣除）")
print("  产品的GMV = 下单金额 - 优惠金额")
print()
print("你的方案（参考）：")
print("  在DWD层定义三个字段：")
print("    gmv_gross    = 下单金额（含未支付、不含退款扣减）")
print("    gmv_net      = 实付金额 - 退款金额")
print("    gmv_net_30d  = 实付金额 - 30天内退款（排除长尾退款）")
print("  所有下游分析统一从DWD/DWS层取数，不再各自从业务库计算。")


In [ ]:
# ============================================================
# 任务4：查询优化
# ============================================================

print("=" * 60)
print("任务4：优化'去年同期的品类分析'（原耗时20分钟）")
print("=" * 60)
print()

print("为什么慢？（结合本周所学分析）")
print("  1. 数据在MySQL中（OLTP）——不适合大查询")
print("  2. 没有分区——全表扫描200亿行")
print("  3. 没有预汇总——每次都要从头算")
print("  4. 没有列裁剪——SELECT * 读了所有列")
print()

print("数仓方案（逐层优化）：")
print("  1. 数据迁移到Hive（OLAP引擎）——允许大数据量分析")
print("  2. 按dt分区——只读相关日期的分区（200亿→1000万行）")
print("  3. DWS层的品类日汇总表——直接查300行而不是5000万行")
print("  4. 列式存储ORC——只读category和amount列")
print()
print("优化链路：MySQL全表扫描200亿行(20分钟)")
print("  → Hive+DWD分区裁剪≈1000万行(1分钟)")
print("  → DWS预聚合=300行(0.1秒)")
print()

print("=" * 60)
print("任务5：数据倾斜预防")
print("=" * 60)
print()
print("识别这个SQL的倾斜风险：")
print("  SELECT user_id, COUNT(DISTINCT product_id), SUM(amount)")
print("  FROM orders")
print("  WHERE dt BETWEEN '2026-01-01' AND '2026-06-30'")
print("  GROUP BY user_id;")
print()
print("风险点：")
print("  1. GROUP BY user_id → 如果有'机器人用户'→严重倾斜")
print("  2. COUNT(DISTINCT product_id) → 去重Shuffle→额外倾斜")
print("  3. 半年的订单量可能数十亿行→数据量大×倾斜=双重瓶颈")
print()
print("优化方案：")
print("  1. 先过滤NULL user_id和已知机器人用户")
print("  2. COUNT(DISTINCT)→先GROUP BY user_id+product_id再COUNT")
print("  3. 如果倾斜仍严重→加盐处理")


In [ ]:
# ============================================================
# 任务6：自评清单
# ============================================================

print("=" * 60)
print("自评：本周核心能力检验")
print("=" * 60)
print()

checks = [
    ("能画出数据从业务系统到分析看板的完整流转路径", "day_01"),
    ("能说出ODS/DWD/DWS/ADS各层的作用并设计分层方案", "day_02"),
    ("能画出简单的星型模型（事实表+维度表）", "day_02"),
    ("能区分SCD Type 1/2/3并选择合适的类型", "day_03"),
    ("能写出Type 2的事实表JOIN维表SQL", "day_03"),
    ("能设计一个有10-15个字段的用户分析宽表", "day_03"),
    ("能解释Hive和MySQL的核心差异（至少5条）", "day_04"),
    ("能写出触发分区裁剪的WHERE条件", "day_04"),
    ("能用MAPJOIN优化大表JOIN小表", "day_04"),
    ("能解释数据倾斜的本质并给出3种处理方法", "day_05"),
    ("能把COUNT(DISTINCT)改写为GROUP BY版本", "day_05"),
    ("能用加盐法处理GROUP BY倾斜", "day_05"),
    ("能设计一个包含分层+SCD+优化+防倾斜的完整数仓方案", "day_06"),
]

for i, (check, day) in enumerate(checks, 1):
    print(f"  [{i:2d}] [{day}] {check}")
    print(f"       通过？(y/n): ")
print()
print("如果全部'y'——恭喜，本周你已经从一个'只会写SQL的分析师'")
print("升级为'有数据工程思维的分析师'。")
print()
print("如果还有'n'——建议回到对应日期的内容重新学习和练习。")


## 📊 区块4：业务场景应用

### 场景1：推动数仓建设的步骤

如果你现在是这个公司的第一个数据分析师，面对上面的数据困境，你怎么推动建设？

**第1步：量化损失**
- 现在团队每周浪费在"等SQL跑完"上的时间：5个分析师 × 每天5次 × 每次10分钟 = 约4小时/天
- 因历史数据被删而无法完成的分析：每月约3次
- 因口径不一致导致的跨部门扯皮：每月约2次

**第2步：提出MVP方案**
- 第一期（1个月）：搭建ODS+DWD层，先把数据从MySQL搬过来，解决"慢"和"被删"的问题
- 第二期（2个月）：搭建DWS层，解决高频查询的性能问题
- 第三期（3个月）：搭建ADS层+看板，解决口径统一问题

**第3步：争取支持**
- 跟CTO说："现在团队每天浪费4小时等SQL——搭建数仓后这个等待时间降到5分钟以内"
- 跟运营总监说："数仓建好后，你看板上的数字和财务的GMV口径将完全一致"
- 跟财务说："历史数据永远不会再被删——你想查三年前的GMV都能查到"

**关键启示**：推动数仓建设不是"技术项目"——它是"ROI项目"。用业务损失来量化需求，用效率提升来说服决策者。

### 场景2：数仓建好了，但没人用——为什么？

数仓花两个月建好了，但团队的分析师还在用原来的MySQL。为什么？

可能的原因：
1. 他们不知道数仓有哪些表（缺少数据字典）
2. 他们不知道怎么在Hive上写SQL（缺少培训）
3. 他们不确定数仓的数据靠不靠谱（缺少信任）
4. Hive比MySQL慢得多（查询延迟——他们不理解MAPJOIN和分区裁剪）

**解决方案**（你作为推动者）：
- 写一份"分析师数仓使用指南"（常用表清单+SQL模板+注意事项）
- 带团队做一次培训（30分钟实操）
- 先选2-3个最常用的分析场景在数仓上跑通，用结果说话
- 告诉团队："Hive不是慢——是你没开分区裁剪"（教他们优化）

### 场景3：数仓不能解决一切——什么情况不能用数仓

有同学学完这周后觉得"数仓能解决一切"——不是的。数仓有它的边界：

**数仓不适合的场景**：
- 实时查询（秒级响应）：应该用ClickHouse/StarRocks
- 在线业务数据处理：应该用MySQL/PostgreSQL
- 小规模数据（<100万行）：数仓"杀鸡用牛刀"——启动MapReduce都要几十秒
- 流式计算（实时可视化）：应该用Flink
- 交互式探索（需要频繁修改SQL）：用Presto/Trino在数仓上做交互层

**数仓适合的场景**：
- T+1日报/周报/月报
- 大规模ETL处理
- 历史数据归档和分析
- 复杂的多表关联分析
- 作为"单一数据真相来源"（Single Source of Truth）

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 能说出一个典型数仓方案包含哪些要素（分层、SCD、分区、命名规范）
- [ ] 能列举至少5张应该在ODS层的表
- [ ] 能区分哪些表适合DWS层、哪些适合ADS层
- [ ] 能说出推动数仓建设的三个关键步骤

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 能独立设计一个部门级（3-5张事实表）的星型模型方案
- [ ] 能根据业务需求为每个维度属性选择正确的SCD类型
- [ ] 能设计合理的分区策略（哪个字段、什么粒度）
- [ ] 能评估数仓方案的查询性能（DWD vs DWS vs ADS）

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能从分析师视角评估一个已有数仓的优缺点
- [ ] 能设计面向新业务线的完整数仓分层方案
- [ ] 能计算数仓建设的ROI（时间节省 × 人数 × 频率）
- [ ] 能在"存储成本"和"查询性能"之间做架构权衡
- [ ] 能识别数仓中长期存在的"反模式"并提出改进方案

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 本周全部** | 数据流转、分层、SCD、Hive、倾斜 | 综合练习是本周全部知识的整合应用 |
| **← 前3周** | 商业思维框架 | 数仓的最终目标是支撑分析——OSM、AARRR、RFM都需要高质量的数据 |
| **→ 第5-6周** | SQL分析实战 | 接下来两周，你要在真实的数仓表上写SQL解决业务问题 |
| **→ 职业发展** | 高级分析师 | 能设计数仓方案的数据分析师，价值远超只会写SQL的 |

### 横向连接

**与数据治理的关联**：
- 数仓是数据治理的基础设施——分层+血缘+质量都需要依托数仓
- 今天设计的方案，本质上就是在做"数据治理的骨架"

**与技术架构的关联**：
- 数仓方案不是孤立的——它需要和公司的技术选型（Hive/Spark/Presto/ClickHouse）协同
- 一个好的分析师能理解技术约束，提出"技术上可实现"的数据方案

## 📝 区块7：当日小结

> **今天是本周知识的"熔炉"——把数据流转、分层建模、SCD设计、Hive优化、倾斜治理全部融合到一个完整的数仓方案中。一个好的数仓设计方案不追求"技术上最先进"，而追求"最适合业务现状"：知道什么时候该建DWS层（高频查询=≥每周5次），知道什么字段用Type 2（需要历史回溯），知道分区用什么字段（首选日期），知道如何推动建设（量化损失+提出MVP+争取支持）。最重要的是：数仓不是目的，是手段——它的价值最终体现在"让分析师少花时间等SQL，多花时间做分析"。**

**三条金句带回家**：
1. 好的数仓设计让分析师从"SQL选手"变成"分析选手"
2. 推动数仓建设不讲技术讲ROI——"每天省4小时"比"我们需要Hive+ORC+分区"更有说服力
3. 数仓不是万能的——它不适合实时、不适合小数据、不适合在线业务

### 🔖 课后拓展

1. **练习**：找到你公司数仓的架构文档（如果没有，试着从表命名倒推分层结构），画出你公司的数仓分层图

2. **思考**：如果公司明年要做"实时数仓"（秒级），今天的T+1离线数仓方案有哪些需要改造？

3. **预习**：下周一进入第5周——SQL分析思维，用本周搭建的数仓表来写真正的业务分析SQL

---

*03analyst_thinking · Week 4 Day 6 · 数仓综合练习*